In [1]:
import pandas as pd

In [2]:
application_df = pd.read_csv(r"C:\Users\priya\Downloads\bank_loan_clean.xls")

In [3]:
application_df.head(3)

,customer_id,target,loan_type,gender,annual_income,loan_amount,loan_installment,goods_price,income_source,education_level,...,days_employed,days_registration,days_id_publish,occupation,employer_type,ext_source_1,ext_source_2,ext_source_3,ext_source_1_missing,age_years
0,100002,1,Cash loans,M,202500.0,406597.5,24700.5,351000.0,Working,Secondary / secondary special,...,-637.0,-3648.0,-2120,Laborers,Business Entity Type 3,0.083037,0.262949,0.139376,0,26.0
1,100003,0,Cash loans,F,270000.0,1293502.5,35698.5,1129500.0,State servant,Higher education,...,-1188.0,-1186.0,-291,Core staff,School,0.311267,0.622246,0.535276,0,46.0
2,100004,0,Revolving loans,M,67500.0,135000.0,6750.0,135000.0,Working,Secondary / secondary special,...,-225.0,-4260.0,-2531,Laborers,Government,0.505998,0.555912,0.729567,1,52.0


In [4]:
application_df.isnull().sum()

customer_id               0
target                    0
loan_type                 0
gender                    0
annual_income             0
loan_amount               0
loan_installment         12
goods_price             278
income_source             0
education_level           0
name_family_status        0
days_employed             0
days_registration         0
days_id_publish           0
occupation                0
employer_type             0
ext_source_1              0
ext_source_2              0
ext_source_3              0
ext_source_1_missing      0
age_years                 0
dtype: int64

In [5]:
application_df.dropna(inplace = True)

In [6]:
app_cols_to_keep = [
    'customer_id', 'target', 'loan_type',
    'gender', 'loan_amount', 'loan_installment',
    'income_source', 'education_level',
    'name_family_status', 'days_employed',
    'occupation', 'ext_source_2', 'ext_source_3',
    'age_years'
]
application_df = application_df[app_cols_to_keep]
print("✅ Main file columns selected!")

✅ Main file columns selected!


In [7]:
previous_df =  pd.read_csv(r"C:\Users\priya\Downloads\previous_loans_clean.xls")

In [8]:
previous_df.head(3)

,previous_loan_id,customer_id,contract_type,annuity_amount,applied_amount,approved_amount,goods_price,contract_status,days_decision
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,17145.0,Approved,-73
1,2802425,108129,Cash loans,25188.615,421650.0,501682.5,607500.0,Approved,-164
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,112500.0,Approved,-301


In [9]:
prev_cols_to_keep = [
    'customer_id', 'applied_amount'
]
previous_df = previous_df[prev_cols_to_keep]
print("✅ Previous file columns selected!")
print(previous_df.columns.tolist())

✅ Previous file columns selected!
['customer_id', 'applied_amount']


In [10]:
merged_df = application_df.merge(previous_df,
                                  on='customer_id',
                                  how='left')
print("Files Merged!")
print(f"Merged shape: {merged_df.shape}")
print(merged_df.columns.tolist())

Files Merged!
Merged shape: (931694, 15)
['customer_id', 'target', 'loan_type', 'gender', 'loan_amount', 'loan_installment', 'income_source', 'education_level', 'name_family_status', 'days_employed', 'occupation', 'ext_source_2', 'ext_source_3', 'age_years', 'applied_amount']


In [11]:
# Drop customer_id
merged_df = merged_df.drop(columns=['customer_id'])

# Fix null values
merged_df['applied_amount']   = merged_df['applied_amount'].fillna(0)
merged_df['loan_installment'] = merged_df['loan_installment'].fillna(
                                merged_df['loan_installment'].median())

# Drop Unknown gender
merged_df = merged_df[merged_df['gender'] != 'Unknown']

print("✅ Done!")
print(f"Shape: {merged_df.shape}")
print(f"Nulls: {merged_df.isnull().sum().sum()}")

✅ Done!
Shape: (931694, 14)
Nulls: 0


In [12]:
# Label Encoding
education_order = {
    'Lower secondary'              : 1,
    'Secondary / secondary special': 2,
    'Incomplete higher'            : 3,
    'Higher education'             : 4,
    'Academic degree'              : 5
}
merged_df['education_level'] = merged_df['education_level'].map(education_order)
print("Label Encoding Done!")

# One Hot Encoding
ohe_cols = ['loan_type', 'gender', 'income_source', 
            'name_family_status', 'occupation']
merged_df = pd.get_dummies(merged_df, columns=ohe_cols, drop_first=True)
print("One Hot Encoding Done!")
print(f"Total columns: {merged_df.shape[1]}")

Label Encoding Done!
One Hot Encoding Done!
Total columns: 41


In [13]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['loan_amount', 'loan_installment', 
              'days_employed', 'ext_source_2', 
              'ext_source_3', 'age_years', 
              'applied_amount']

scaler = StandardScaler()
merged_df[scale_cols] = scaler.fit_transform(merged_df[scale_cols])
print("✅ Scaling Done!")

✅ Scaling Done!


In [14]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

In [15]:
# SMOTE
X = merged_df.drop(columns=['target'])
y = merged_df['target']

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
print("✅ SMOTE Done!")
print(f"Before SMOTE: {y.value_counts().to_dict()}")
print(f"After SMOTE : {y_resampled.value_counts().to_dict()}")
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
                                    X_resampled,
                                    y_resampled,
                                    test_size=0.2,
                                    random_state=42)
print("✅ Train Test Split Done!")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")

# Save to Avoid Future Loss!

X_resampled.to_csv('X_resampled.csv', index=False)
y_resampled.to_csv('y_resampled.csv', index=False)
print("✅ Data Saved!")

✅ SMOTE Done!
Before SMOTE: {0: 851587, 1: 80107}
After SMOTE : {1: 851587, 0: 851587}
✅ Train Test Split Done!
X_train: (1362539, 40)
X_test : (340635, 40)
✅ Data Saved!


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,
                             classification_report)

model_lr = LogisticRegression(random_state=42, max_iter=1000)
model_lr.fit(X_train, y_train)
print("✅ Model Trained!")

y_pred_lr = model_lr.predict(X_test)
print(f"\n🎯 Accuracy: {round(accuracy_score(y_test, y_pred_lr) * 100, 2)}%")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_lr))

✅ Model Trained!

🎯 Accuracy: 67.51%

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.69      0.68    170244
           1       0.68      0.66      0.67    170391

    accuracy                           0.68    340635
   macro avg       0.68      0.68      0.68    340635
weighted avg       0.68      0.68      0.68    340635



In [17]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1)

model_rf.fit(X_train, y_train)
print("✅ Model Trained!")

y_pred_rf = model_rf.predict(X_test)
print(f"\n🎯 Accuracy: {round(accuracy_score(y_test, y_pred_rf) * 100, 2)}%")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_rf))

✅ Model Trained!

🎯 Accuracy: 98.54%

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99    170244
           1       0.99      0.98      0.99    170391

    accuracy                           0.99    340635
   macro avg       0.99      0.99      0.99    340635
weighted avg       0.99      0.99      0.99    340635



In [18]:
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,      # ← limit tree depth!
    min_samples_leaf=50, # ← minimum samples per leaf
    random_state=42,
    n_jobs=-1)

model_rf.fit(X_train, y_train)
print("✅ Model Trained!")

y_pred_rf = model_rf.predict(X_test)
print(f"\n🎯 Accuracy: {round(accuracy_score(y_test, y_pred_rf) * 100, 2)}%")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_rf))

✅ Model Trained!

🎯 Accuracy: 71.25%

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.72      0.72    170244
           1       0.72      0.70      0.71    170391

    accuracy                           0.71    340635
   macro avg       0.71      0.71      0.71    340635
weighted avg       0.71      0.71      0.71    340635



In [19]:
!pip install xgboost


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1)

model_xgb.fit(X_train, y_train)
print("✅ Model Trained!")

y_pred_xgb = model_xgb.predict(X_test)
print(f"\n🎯 Accuracy: {round(accuracy_score(y_test, y_pred_xgb) * 100, 2)}%")
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred_xgb))

✅ Model Trained!

🎯 Accuracy: 74.46%

📊 Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.75      0.75    170244
           1       0.75      0.74      0.74    170391

    accuracy                           0.74    340635
   macro avg       0.74      0.74      0.74    340635
weighted avg       0.74      0.74      0.74    340635



In [21]:
# Predict probability of being risky
risk_probability = model_xgb.predict_proba(X_test)

# Get risk % for risky class (1)
risk_percent = risk_probability[:, 1] * 100

# See sample predictions
for i in range(5):
    print(f"Customer {i+1} → Risk: {risk_percent[i]:.2f}%")

Customer 1 → Risk: 74.03%
Customer 2 → Risk: 79.26%
Customer 3 → Risk: 56.05%
Customer 4 → Risk: 14.46%
Customer 5 → Risk: 54.74%


In [22]:
# Add risk categories
def risk_category(risk):
    if risk >= 70:
        return '🔴 High Risk'
    elif risk >= 40:
        return '🟡 Medium Risk'
    else:
        return '🟢 Low Risk'

for i in range(5):
    category = risk_category(risk_percent[i])
    print(f"Customer {i+1} → {risk_percent[i]:.2f}% → {category}")

Customer 1 → 74.03% → 🔴 High Risk
Customer 2 → 79.26% → 🔴 High Risk
Customer 3 → 56.05% → 🟡 Medium Risk
Customer 4 → 14.46% → 🟢 Low Risk
Customer 5 → 54.74% → 🟡 Medium Risk


In [23]:
import pickle

# Save XGBoost model
with open('loan_risk_model.pkl', 'wb') as f:
    pickle.dump(model_xgb, f)

# Save scaler too — needed for new predictions!
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Model Saved!")
print("✅ Scaler Saved!")

✅ Model Saved!
✅ Scaler Saved!


In [24]:
# Save exact column names model was trained on
import pickle

train_columns = X_train.columns.tolist()

with open('C:/Users/priya/Downloads/train_columns.pkl', 'wb') as f:
    pickle.dump(train_columns, f)

with open('C:/Users/priya/Downloads/loan_risk_model.pkl', 'wb') as f:
    pickle.dump(model_xgb, f)

with open('C:/Users/priya/Downloads/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ All files saved!")
print(f"Total training columns: {len(train_columns)}")
print(train_columns)

✅ All files saved!
Total training columns: 40
['loan_amount', 'loan_installment', 'education_level', 'days_employed', 'ext_source_2', 'ext_source_3', 'age_years', 'applied_amount', 'loan_type_Revolving loans', 'gender_M', 'gender_XNA', 'income_source_Commercial associate', 'income_source_Maternity leave', 'income_source_Pensioner', 'income_source_State servant', 'income_source_Student', 'income_source_Unemployed', 'income_source_Working', 'name_family_status_Married', 'name_family_status_Separated', 'name_family_status_Single / not married', 'name_family_status_Widow', 'occupation_Cleaning staff', 'occupation_Cooking staff', 'occupation_Core staff', 'occupation_Drivers', 'occupation_HR staff', 'occupation_High skill tech staff', 'occupation_IT staff', 'occupation_Laborers', 'occupation_Low-skill Laborers', 'occupation_Managers', 'occupation_Medicine staff', 'occupation_Private service staff', 'occupation_Realty agents', 'occupation_Sales staff', 'occupation_Secretaries', 'occupation_Se

In [25]:
!pip install streamlit


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
